In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
df_allowed = pd.read_csv("../../datasets/allowed_qualifiers_repairs.csv")

df_allowed

In [ ]:
len(df_allowed[(df_allowed['Constraint Deleted'] == True)] )

In [ ]:
len(df_allowed[(df_allowed['Constraint Deprecated'] == True)] )

In [ ]:
len(df_allowed[(df_allowed['Included as Exception'] == True)] )

In [ ]:
len(df_allowed[(df_allowed['T-box qualifier added to constraint'] == True)] )

In [ ]:
df_allowed[(df_allowed['Constraint Deleted'] == False) & 
     (df_allowed['Constraint Deprecated'] == False)& 
     (df_allowed['Included as Exception'] == False)& 
     (df_allowed['T-box qualifier added to constraint'] == False) 
     
    ]

In [ ]:
df_allowed["A-box statement Deleted"] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemoved(subject, wd_prop, stmt):
    # URL of the endpoint - wikidata 2023
    endpoint = "ENTER_qEndpoint_WD_2023"

    p_prop = wd_prop.replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/")
    # SPARQL query
    query = f"""ASK {{ <{subject}><{p_prop}><{stmt}>    }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
subject = "http://www.wikidata.org/entity/P5368"
wd_p = "http://www.wikidata.org/entity/P2302"
stmt = "http://www.wikidata.org/entity/statement/P5368-98732398-43f4-45c2-6684-06e5ffb59d7f"
print(isRemoved(subject,wd_p, stmt))


In [ ]:
for index, row in df_allowed.iterrows():
    
    if (index % 10000 == 0):
        print(index)
    
    if row['A-box statement Deleted'] is None:
        df_allowed.at[index, 'A-box statement Deleted'] = isRemoved(row['subject'], row['wd_property'], row["instance_statement"])
        

In [ ]:
df_allowed

In [ ]:
df_allowed.iloc[4]["instance_statement"]

In [ ]:
len(df_allowed[(df_allowed['A-box statement Deleted'] == True)] )

In [ ]:
df_allowed.to_csv("allowed_qualifiers_repairs.csv", index=False)

In [ ]:
df_allowed[(df_allowed['Constraint Deleted'] == False) & 
     (df_allowed['Constraint Deprecated'] == False)& 
     (df_allowed['Included as Exception'] == False)& 
     (df_allowed['T-box qualifier added to constraint'] == False) & 
     (df_allowed['A-box statement Deleted'] == False) 
     
    ]

In [ ]:
df_allowed.iloc[1]["instance_statement"]

In [ ]:
df_allowed.iloc[1]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def get_constraint_instance(wd_property, pq_qualifier):
    # URL of the endpoint - wikidata 2019
    endpoint = "ENTER_qEndpoint_WD_2019"

    # SPARQL query
    query = f"""PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
                PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
                PREFIX wikibase: <http://wikiba.se/ontology#>
                PREFIX p: <http://www.wikidata.org/prop/>
                PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
                PREFIX ps: <http://www.wikidata.org/prop/statement/>
                PREFIX wd: <http://www.wikidata.org/entity/>
                PREFIX wdt: <http://www.wikidata.org/prop/direct/>

                SELECT
                  ?constraint_statement
                WHERE
                {{
                  <{wd_property}> p:P2302 ?constraint_statement.
                  ?constraint_statement ps:P2302 wd:Q21510851. ## allowed qualifiers constraint

                  ?wd_qualifier wikibase:qualifier <{pq_qualifier}>.
                  FILTER NOT EXISTS {{?constraint_statement pq:P2306 ?wd_qualifier}}
                }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        constraint_statement = []
        for binding in root.findall('.//ns:binding[@name="constraint_statement"]/ns:uri', namespace):
            constraint_statement.append(binding.text)

        return constraint_statement[0]
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    return False

# Example usage
wd_property = "http://www.wikidata.org/entity/P2302"
pq_qualifier = "http://www.wikidata.org/prop/qualifier/P2308"
print(get_constraint_instance(wd_property, pq_qualifier))


In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemovedConstraint(constraint_statement):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK
                {{
                  <{constraint_statement}> ?p ?o.
                }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    return False

# Example usage
wd_property = "http://www.wikidata.org/entity/P2302"
pq_qualifier = "http://www.wikidata.org/prop/qualifier/P2308"
print(isRemovedConstraint(get_constraint_instance(wd_property, pq_qualifier)))


In [ ]:
df_allowed_copy = df_allowed

In [ ]:
df_allowed_copy

In [ ]:
len(df_allowed_copy[(df_allowed_copy['Constraint Deleted'] == True)] )

In [ ]:
df_allowed_copy['Constraint Deleted'].value_counts()

In [ ]:
for index, row in df_allowed_copy.iterrows():
    
    if (index % 10000 == 0):
        print(index)
    
    if row['Constraint Deleted'] is False:
        df_allowed_copy.at[index, 'Constraint Deleted'] = isRemovedConstraint(
            get_constraint_instance(row['wd_property'], row['not_allowed_qualifier'])
        )
        

In [ ]:
len(df_allowed_copy[(df_allowed_copy['Constraint Deleted'] == True)] )

In [ ]:
df_allowed_copy.dtypes

In [ ]:
df_allowed_copy[(df_allowed_copy['Constraint Deleted'] == False) & 
     (df_allowed_copy['Constraint Deprecated'] == False)& 
     (df_allowed_copy['Included as Exception'] == False)& 
     (df_allowed_copy['T-box qualifier added to constraint'] == False) & 
     (df_allowed_copy['A-box statement Deleted'] == False) 
    ]

In [ ]:
df_allowed_copy.iloc[15]["instance_statement"]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemovedQualifier(instance_statement, not_allowed_qualifier):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK
                {{
                  <{instance_statement}> <{not_allowed_qualifier}> ?o.
                }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    return False
 
# Example usage
instance_statement = "http://www.wikidata.org/entity/statement/Q62637265-49b2b88b-f9ba-4f15-abd5-2701206a6cf5"
not_allowed_qualifier = "http://www.wikidata.org/prop/qualifier/P4100"
print(isRemovedQualifier(instance_statement, not_allowed_qualifier))


In [ ]:
df_allowed_copy["A-box non-allowed qua removed"] = None

In [ ]:
for index, row in df_allowed_copy.iterrows():
    
    if (index % 10000 == 0):
        print(index)
    
    if row['A-box non-allowed qua removed'] is None:
        if row["A-box statement Deleted"] is False:
            df_allowed_copy.at[index, 'A-box non-allowed qua removed'] = isRemovedQualifier(row['instance_statement'], row['not_allowed_qualifier'])
        else:
            df_allowed_copy.at[index, 'A-box non-allowed qua removed'] = False
        

In [ ]:
df_allowed_copy.to_csv("allowed_qualifiers_repairs_final.csv", index=False)

In [ ]:
df_allowed_copy

In [ ]:
len(df_allowed_copy[(df_allowed_copy['A-box non-allowed qua removed'] == True)] )

In [ ]:
df_allowed_copy[(df_allowed_copy['Constraint Deleted'] == False) & 
     (df_allowed_copy['Constraint Deprecated'] == False)& 
     (df_allowed_copy['Included as Exception'] == False)& 
     (df_allowed_copy['T-box qualifier added to constraint'] == False) & 
     (df_allowed_copy['A-box statement Deleted'] == False) & 
     (df_allowed_copy['A-box non-allowed qua removed'] == False) 
    ]

In [ ]:
df_allowed_copy.dtypes

In [ ]:

df_allowed_copy['A-box statement Deleted'] = df_allowed_copy['A-box statement Deleted'].astype(bool)

In [ ]:
df_allowed_copy['A-box non-allowed qua removed'] = df_allowed_copy['A-box non-allowed qua removed'].astype(bool)

In [ ]:
df_allowed_copy.dtypes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Create the new DataFrame with the required columns
df2 = pd.DataFrame()
df2['A-box changes'] = df_allowed_copy['A-box statement Deleted'] | df_allowed_copy['A-box non-allowed qua removed']
df2['T-box changes'] = (
    df_allowed_copy['Constraint Deleted'] | 
    df_allowed_copy['Constraint Deprecated'] | 
    df_allowed_copy['Included as Exception'] | 
    df_allowed_copy['T-box qualifier added to constraint']
)

# Calculate the sizes of the sets
a_box_changes = df2['A-box changes'].sum()
t_box_changes = df2['T-box changes'].sum()
intersection = (df2['A-box changes'] & df2['T-box changes']).sum()

# Plot the Venn diagram
venn2(subsets=(a_box_changes, t_box_changes, intersection), 
      set_labels=('A-box changes', 'T-box changes'))


# Add title
plt.title("Allowed Qualifiers Constraint repairs")

plt.show()
